## Prerequisite Code

In [0]:
# Import required libraries
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
# Additional required libraries
from pyspark.sql.window import Window

In [0]:
%run ../initial-setup/03-utils

In [0]:
# Create widgets
dbutils.widgets.text('catalog', 'sportsdirect_sales', 'Catalog')
dbutils.widgets.text('data_source', 'gross_price', 'Data Source')

# Access widgets
catalog = dbutils.widgets.get('catalog')
data_source = dbutils.widgets.get('data_source')

In [0]:
# Define source directory
source_dir = f's3://sd-warrior-acquisition/{data_source}/*.csv'

## Warrior Bronze Layer

In [0]:
# Read data from the source directory
raw_data = spark.read \
    .format('csv') \
    .option('header', True) \
    .option('inferSchema', True) \
    .load(source_dir) \
    .withColumn('read_timestamp', F.current_timestamp()) \
    .select('*', '_metadata.file_name', '_metadata.file_size')

# View raw data
display(raw_data)

product_id,month,gross_price,read_timestamp,file_name,file_size
25891101,2025/07/01,-84,2026-03-18T04:55:10.044Z,gross_price.csv,2741
25891101,01/08/2025,unknown,2026-03-18T04:55:10.044Z,gross_price.csv,2741
25891101,2025/09/01,84,2026-03-18T04:55:10.044Z,gross_price.csv,2741
25891101,2025-10-01,83,2026-03-18T04:55:10.044Z,gross_price.csv,2741
25891101,2025-11-01,83,2026-03-18T04:55:10.044Z,gross_price.csv,2741
88888888,2025-12-01,-83,2026-03-18T04:55:10.044Z,gross_price.csv,2741
25891102,2025-07-01,68,2026-03-18T04:55:10.044Z,gross_price.csv,2741
25891102,2025-08-01,68,2026-03-18T04:55:10.044Z,gross_price.csv,2741
25891102,2025-09-01,68,2026-03-18T04:55:10.044Z,gross_price.csv,2741
25891102,2025-10-01,69,2026-03-18T04:55:10.044Z,gross_price.csv,2741


In [0]:
# Write raw data to the bronze table
raw_data.write \
    .format('delta') \
    .option('enableChangeDataFeed', True) \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{wr_bronze_schema}.dim_price')

## Warrior Silver Layer

In [0]:
# Clean data and apply transformations

# Fix month format
transformed_data = raw_data \
    .withColumn(
        'month',
        F.coalesce(
            F.try_to_date(F.col('month'), 'yyyy/MM/dd'),
            F.try_to_date(F.col('month'), 'dd/MM/yyyy'),
            F.try_to_date(F.col('month'), 'yyyy-MM-dd'),
            F.try_to_date(F.col('month'), 'dd-MM-yyyy')
        )
    )

# Fix prices
transformed_data = transformed_data \
    .withColumn(
        'gross_price',
        F.when(
            F.col('gross_price').rlike('^-?\d+(\.\d+)?$'),
                F.when(
                    F.col('gross_price').cast('double') < 0, -1 * F.col('gross_price').cast('double')
                )
                .otherwise(F.col('gross_price').cast('double'))
        )
        .otherwise(0)
    )

# Rename gross_price to price_inr
transformed_data = transformed_data. \
    withColumnRenamed(
        'gross_price',
        'price_inr'
    )

# Add product_codes for prices with valid product_ids
dim_product_table = spark.table(f'{catalog}.{wr_silver_schema}.dim_product')

transformed_data = transformed_data. \
    join(
        dim_product_table.select('product_id', 'product_code'),
        on='product_id',
        how='inner'
    ) \
    .select('product_id', 'product_code', 'month', 'price_inr', 'read_timestamp', 'file_name', 'file_size')

<>:20: SyntaxWarning: invalid escape sequence '\d'
<>:20: SyntaxWarning: invalid escape sequence '\d'
/home/spark-2e1700d4-ae90-4f05-b667-3c/.ipykernel/2706/command-6481370314544712-3801857592:20: SyntaxWarning: invalid escape sequence '\d'
  F.col('gross_price').rlike('^-?\d+(\.\d+)?$'),


In [0]:
# Verify month format
display(transformed_data.select('month').distinct())

month
2025-08-01
2025-11-01
2025-10-01
2025-12-01
2025-07-01
2025-09-01


In [0]:
# Check prices
display(transformed_data.filter(F.col('price_inr') < 0).count())

0

In [0]:
# Verify product codes
display(transformed_data)

product_id,product_code,month,price_inr,read_timestamp,file_name,file_size
25891302,81e307407f86ff25dddaa1b083aa284c5d209005a2754782cfe62bc07ff5774c,2025-12-01,300.0,2026-03-18T04:55:40.518Z,gross_price.csv,2741
25891303,2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5,2025-12-01,432.0,2026-03-18T04:55:40.518Z,gross_price.csv,2741
25891503,c3a270caf0285f44bbd624e9a209a46e86f47adf81d2bcadf5969ff260d97d12,2025-12-01,281.0,2026-03-18T04:55:40.518Z,gross_price.csv,2741
25891403,8395d734ede7e81a35c67ea3ae7240db0380f8370fb488a3872ea033af705ee4,2025-12-01,50.0,2026-03-18T04:55:40.518Z,gross_price.csv,2741
25891401,958f41cd549041a991f52c5080db909845931526051f6a535e0221223259b5e2,2025-12-01,79.0,2026-03-18T04:55:40.518Z,gross_price.csv,2741
25891101,95dd546ad1c0e319431aabb5d05da6af9d0418dbf7decda178337b8c26cc898f,2025-11-01,83.0,2026-03-18T04:55:40.518Z,gross_price.csv,2741
25891102,00d0ee2f06cdd39f74be341dac1321e96d6810a2a95de468c7ce987e4de4121f,2025-12-01,69.0,2026-03-18T04:55:40.518Z,gross_price.csv,2741
25891601,19f73e3f8a3942a7aaa3d1a7b46cf31314589ba429ff4f9d2b8a047203f3eceb,2025-11-01,440.0,2026-03-18T04:55:40.518Z,gross_price.csv,2741
25891402,41d9f8038c4771bf55fdddeaf9e940f5e23b717d63c6a992e2654afc37fc2c8d,2025-12-01,0.0,2026-03-18T04:55:40.518Z,gross_price.csv,2741
25891201,2e73a3ac7f86e06279f1d690401af67a2c6109457d025f9b0f459bd708a60689,2025-12-01,108.0,2026-03-18T04:55:40.518Z,gross_price.csv,2741


In [0]:
# Write transformed data to the silver table
transformed_data.write \
    .format('delta') \
    .option('enableChangeDataFeed', 'true') \
    .option('mergeSchema', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{wr_silver_schema}.dim_price')

## Warrior Gold Layer

In [0]:
# Get analytics data from the silver layer
analytics_data = transformed_data.select('product_code', 'price_inr', 'month')

# View analytics data
display(analytics_data)

product_code,price_inr,month
81e307407f86ff25dddaa1b083aa284c5d209005a2754782cfe62bc07ff5774c,300.0,2025-12-01
2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5,432.0,2025-12-01
c3a270caf0285f44bbd624e9a209a46e86f47adf81d2bcadf5969ff260d97d12,281.0,2025-12-01
8395d734ede7e81a35c67ea3ae7240db0380f8370fb488a3872ea033af705ee4,50.0,2025-12-01
958f41cd549041a991f52c5080db909845931526051f6a535e0221223259b5e2,79.0,2025-12-01
95dd546ad1c0e319431aabb5d05da6af9d0418dbf7decda178337b8c26cc898f,83.0,2025-11-01
00d0ee2f06cdd39f74be341dac1321e96d6810a2a95de468c7ce987e4de4121f,69.0,2025-12-01
19f73e3f8a3942a7aaa3d1a7b46cf31314589ba429ff4f9d2b8a047203f3eceb,440.0,2025-11-01
41d9f8038c4771bf55fdddeaf9e940f5e23b717d63c6a992e2654afc37fc2c8d,0.0,2025-12-01
2e73a3ac7f86e06279f1d690401af67a2c6109457d025f9b0f459bd708a60689,108.0,2025-12-01


In [0]:
# Write analytics data to the gold table
analytics_data.write \
    .format('delta') \
    .option('enableChangeDataFeed', 'true') \
    .mode('overwrite') \
    .saveAsTable(f'{catalog}.{wr_gold_schema}.dim_price')

## Sports Direct Gold Layer

In [0]:
# Match the Warrior analytics data with Sports Direct analytics data

# Calculate each product price per year

# Add a year column
analytics_data = analytics_data. \
    withColumn(
        'year',
        F.year(F.col('month'))
    )

# Remove records with no price_inr
analytics_data = analytics_data.filter(F.col('price_inr') > 0)

# Define a window
w = Window.partitionBy('product_code', 'year').orderBy(F.col('month').desc())

# Get the latest price for each product per year
analytics_data = analytics_data \
    .withColumn(
        'rank',
        F.row_number().over(w)
    ) \
    .filter(
        F.col('rank') == 1
    )

# Convert price_inr to int
analytics_data = analytics_data \
    .withColumn(
        'price_inr',
        F.col('price_inr').cast('int')
    )

# Finalize analytics data
analytics_data = analytics_data.select('product_code', 'price_inr', 'year')

In [0]:
# Verify final analytics data
display(analytics_data)

product_code,price_inr,year
00d0ee2f06cdd39f74be341dac1321e96d6810a2a95de468c7ce987e4de4121f,69,2025
19f73e3f8a3942a7aaa3d1a7b46cf31314589ba429ff4f9d2b8a047203f3eceb,440,2025
2c41f5528e8561db1cc03bbefa0735a666d8349cb52166efa1562ddc55cd0ee5,432,2025
2e73a3ac7f86e06279f1d690401af67a2c6109457d025f9b0f459bd708a60689,108,2025
41d9f8038c4771bf55fdddeaf9e940f5e23b717d63c6a992e2654afc37fc2c8d,53,2025
62254dca28e1f3ce45668a4abc8571ad4fc3923df56d7a0c12c369dc76aa1f67,100,2025
6a18f762edaea4192d8a27e46560c29aecf7b2689792172da9a7c1c4b5532909,86,2025
798a750ff46551522957fee953cee201003fd974698c192efb3e4649be4c9652,187,2025
7cde4fee80f465659932d7e9336957d070865e23db86fb63dd73a40dc309f430,138,2025
81e307407f86ff25dddaa1b083aa284c5d209005a2754782cfe62bc07ff5774c,300,2025


In [0]:
# Merge the final Warrior analytics data to the Sports Direct analytics data

sd_dim_price_table = DeltaTable.forName(spark, f'{catalog}.{sd_gold_schema}.dim_price')

sd_dim_price_table.alias('target') \
    .merge(
        analytics_data.alias('source'),
        'target.product_code = source.product_code'
    )\
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]